# Reference-ligand redocking validation -- AChE (1EVE), AutoDock Vina

Standalone notebook. Validates the docking protocol used for compound screening against **AChE (PDB 1EVE)** by
redocking the co-crystallized ligand **E20 (donepezil)** and comparing the docked poses with the crystal pose.

**Protocol identical to the screening notebooks** (`1_-Molecular_Docking_1EVE_*.ipynb`): same receptor
(`1EVE_clean_H.pdbqt`, LePro + AutoDockTools `prepare_receptor`), same box (native ligand + 5.0 Å, Jupyter_Dock `getbox`),
AutoDock Vina 1.2 with the `vina` scoring function, exhaustiveness 10, 10 poses, seed 42.

**RMSD is computed in place** (`rdMolAlign.CalcRMS`: heavy atoms, symmetry-aware, *no superposition*), i.e. in the
binding-site frame, which is the standard redocking criterion (success: RMSD < 2.0 Å). The earlier redocking cell in the
screening notebooks used `GetBestRMS`, which first superimposes the pose onto the crystal ligand and therefore ignores where
the pose is in the site; that value is still reported here, for comparison only. Bond orders are assigned from the
ligand's SMILES template, and an element-based MCS in-place RMSD is used as fallback if the template match fails
(this is what made the RMSD uncomputable for 2FZD before).

**Contents:** 1. Setup -- 2. Fetch & clean (skipped if files exist) -- 3. Receptor -- 4. Docking box -- 5. Native ligand --
6. Redocking -- 7. RMSD vs crystal pose -- 8. Seed robustness -- 9. Result for Table 15

In [1]:
import os, sys, glob, shutil, subprocess
import numpy as np, pandas as pd
from pymol import cmd
from vina import Vina
from openbabel import openbabel as ob
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, rdMolAlign, rdFMCS
import warnings
warnings.filterwarnings("ignore"); RDLogger.DisableLog("rdApp.*")

# ---- Jupyter_Dock utilities (same getbox as the screening notebooks) ----
JD = "/home/ali/medical plants/docking/Jupyter_Dock-main"
sys.path.insert(1, os.path.join(JD, "utilities"))
try:
    from utils import getbox
    print("Using Jupyter_Dock getbox()")
except ImportError:                       # identical arithmetic, used only if utils is not on the path
    def getbox(selection="sele", extending=6.0, software="vina"):
        ([minX, minY, minZ], [maxX, maxY, maxZ]) = cmd.get_extent(selection)
        minX, minY, minZ = minX - extending, minY - extending, minZ - extending
        maxX, maxY, maxZ = maxX + extending, maxY + extending, maxZ + extending
        size   = {"size_x": maxX - minX, "size_y": maxY - minY, "size_z": maxZ - minZ}
        center = {"center_x": (maxX + minX) / 2, "center_y": (maxY + minY) / 2, "center_z": (maxZ + minZ) / 2}
        return center, size
    print("Jupyter_Dock utils not found -- using built-in getbox() (same formula)")

LEPRO_BIN    = os.path.join(JD, "bin", "lepro_linux_x86")
PREP_REC_BIN = os.path.join(JD, "bin", "prepare_receptor")
PREP_LIG_BIN = os.path.join(JD, "bin", "prepare_ligand")
for b in (LEPRO_BIN, PREP_REC_BIN, PREP_LIG_BIN):
    print(("found    " if os.path.exists(b) else "MISSING  ") + b)

Using Jupyter_Dock getbox()
found    /home/ali/medical plants/docking/Jupyter_Dock-main/bin/lepro_linux_x86
found    /home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_receptor
found    /home/ali/medical plants/docking/Jupyter_Dock-main/bin/prepare_ligand


In [2]:
# ---------------- configuration ----------------
DOCKING_DIR = "/home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/"   # same folder as screening
PDB_CODE    = "1EVE"
TARGET      = "AChE"
NATIVE_SEL  = "resn E20"           # co-crystallized ligand (E20 (donepezil))
NATIVE_SMILES = "COc1cc2c(cc1OC)C(=O)C(CC1CCN(Cc3ccccc3)CC1)C2"   # bond-order template

VINA_SEED, EXHAUSTIVENESS, N_POSES = 42, 10, 10        # identical to screening
BOX_EXTENDING = 5.0
RMSD_CUTOFF   = 2.0
ADD_HYDROGENS = True     # protonate the crystal ligand at pH 7.4 (screening ligands carry H too)
SEEDS_CHECK   = [42, 1, 7, 123, 2024]                  # section 8

# Box reported in the paper (Table 16) -- used only to check that the same box is recomputed
PAPER_CENTER = (2.0225, 63.295, 67.0615)
PAPER_SIZE   = (16.913, 21.296, 21.733)
PAPER_RMSD   = "0.812 Å (Table 15, value from the earlier GetBestRMS cell)"

os.makedirs(DOCKING_DIR, exist_ok=True)
REDOCK_DIR = os.path.join(DOCKING_DIR, "redocking_validation", PDB_CODE)
os.makedirs(REDOCK_DIR, exist_ok=True)
os.chdir(DOCKING_DIR)

CLEAN_PDB      = os.path.join(DOCKING_DIR, f"{PDB_CODE}_clean.pdb")
RECEPTOR_PDB   = os.path.join(DOCKING_DIR, f"{PDB_CODE}_clean_H.pdb")
RECEPTOR_PDBQT = os.path.join(DOCKING_DIR, f"{PDB_CODE}_clean_H.pdbqt")
NATIVE_LIG     = os.path.join(DOCKING_DIR, f"{PDB_CODE}_native_lig.mol2")
print("Working folder :", DOCKING_DIR)
print("Redock outputs :", REDOCK_DIR)

Working folder : /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/
Redock outputs : /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/redocking_validation/1EVE


## 2. Fetch and clean the structure

Runs only if the files from the screening notebooks are missing, so the validated receptor is reused unchanged.

In [3]:
if not (os.path.exists(CLEAN_PDB) and os.path.exists(NATIVE_LIG)):
    print(f"Fetching {PDB_CODE} ...")
    cmd.fetch(code=PDB_CODE, type="pdb1")
    cmd.select(name="Prot", selection="polymer.protein")
    cmd.save(filename=CLEAN_PDB, format="pdb", selection="Prot", state=1)
    cmd.select(name="NativeLig", selection=NATIVE_SEL)
    cmd.save(filename=NATIVE_LIG, format="mol2", selection="NativeLig", state=1)
    cmd.delete("all")
    for f in glob.glob(f"{PDB_CODE.lower()}*.pdb*") + glob.glob(f"{PDB_CODE}*.cif"):
        if os.path.abspath(f) not in (CLEAN_PDB,): os.remove(f)     # remove the raw download
else:
    print("Reusing existing files from the screening notebooks")

cmd.load(NATIVE_LIG, "natlig")
_m = cmd.get_model("natlig")
print(f"Native ligand: {sorted(set((a.resn, a.resi, a.chain) for a in _m.atom))}, "
      f"{sum(a.symbol != 'H' for a in _m.atom)} heavy atoms "
      f"(template {Chem.MolFromSmiles(NATIVE_SMILES).GetNumHeavyAtoms()})")
cmd.delete("all")

Fetching 1EVE ...
Native ligand: [('E20', '2001', 'A')], 28 heavy atoms (template 28)


## 3. Receptor (same preparation as screening: LePro → `prepare_receptor`)

In [4]:
def run(cmdline):
    r = subprocess.run(cmdline, capture_output=True, text=True, shell=isinstance(cmdline, str))
    return r.returncode, (r.stdout + r.stderr)[-600:]

if not os.path.exists(RECEPTOR_PDBQT):
    if not os.path.exists(RECEPTOR_PDB):
        if os.path.exists(LEPRO_BIN):
            print("LePro ..."); run([LEPRO_BIN, CLEAN_PDB]); os.rename("pro.pdb", RECEPTOR_PDB)
        else:
            print("LePro not found -- adding hydrogens with OpenBabel at pH 7.4 (fallback)")
            run(["obabel", CLEAN_PDB, "-O", RECEPTOR_PDB, "-h", "-p", "7.4"])
    if os.path.exists(PREP_REC_BIN):
        print("prepare_receptor ..."); run([PREP_REC_BIN, "-v", "-r", RECEPTOR_PDB, "-o", RECEPTOR_PDBQT])
    if not os.path.exists(RECEPTOR_PDBQT) or os.path.getsize(RECEPTOR_PDBQT) == 0:
        print("prepare_receptor unavailable/failed -- OpenBabel -xr fallback")
        run(["obabel", RECEPTOR_PDB, "-O", RECEPTOR_PDBQT, "-xr"])
else:
    print("Reusing", RECEPTOR_PDBQT)
assert os.path.getsize(RECEPTOR_PDBQT) > 0
n_atoms = sum(1 for l in open(RECEPTOR_PDBQT) if l.startswith(("ATOM", "HETATM")))
print(f"Receptor PDBQT: {n_atoms} atoms")

LePro ...
prepare_receptor ...
Receptor PDBQT: 5162 atoms


## 4. Docking box (native ligand + 5.0 Å) -- checked against Table 16

In [5]:
cmd.load(RECEPTOR_PDB, "prot"); cmd.load(NATIVE_LIG, "natlig")
center, size = getbox(selection="natlig", extending=BOX_EXTENDING, software="vina")
cmd.delete("all")
C = [center["center_x"], center["center_y"], center["center_z"]]
S = [size["size_x"], size["size_y"], size["size_z"]]
print("center:", [round(v, 4) for v in C], "\nsize  :", [round(v, 4) for v in S])
dc = max(abs(a - b) for a, b in zip(C, PAPER_CENTER)); ds = max(abs(a - b) for a, b in zip(S, PAPER_SIZE))
print(f"Max difference from Table 16: centre {dc:.4f} Å, size {ds:.4f} Å ->",
      "same box as the paper" if dc < 0.01 and ds < 0.01 else "CHECK: box differs from Table 16")

center: [2.0225, 63.295, 67.0615] 
size  : [16.913, 21.296, 21.733]
Max difference from Table 16: centre 0.0000 Å, size 0.0000 Å -> same box as the paper


## 5. Native ligand (crystal geometry → PDBQT)

In [6]:
def obconvert(src, dst, add_h_ph=None):
    conv = ob.OBConversion()
    conv.SetInAndOutFormats(os.path.splitext(src)[1][1:], os.path.splitext(dst)[1][1:])
    mol = ob.OBMol(); conv.ReadFile(mol, src)
    if add_h_ph is not None:
        mol.CorrectForPH(add_h_ph); mol.AddHydrogens(False, True, add_h_ph)
    conv.WriteFile(mol, dst)
    return mol.NumAtoms()

native_h_pdb = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_H.pdb")
native_pdbqt = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_redock.pdbqt")
n = obconvert(NATIVE_LIG, native_h_pdb, add_h_ph=7.4 if ADD_HYDROGENS else None)
print(f"Native ligand with hydrogens: {n} atoms ({'pH 7.4' if ADD_HYDROGENS else 'no H added'})")

if os.path.exists(PREP_LIG_BIN):
    run([PREP_LIG_BIN, "-v", "-l", native_h_pdb, "-o", native_pdbqt])
if not os.path.exists(native_pdbqt) or os.path.getsize(native_pdbqt) == 0:
    obconvert(native_h_pdb, native_pdbqt)
tors = sum(1 for l in open(native_pdbqt) if l.startswith("BRANCH"))
print(f"Ligand PDBQT ready: {native_pdbqt}  ({tors} rotatable branches)")

Native ligand with hydrogens: 58 atoms (pH 7.4)
Ligand PDBQT ready: /home/ali/Desktop/Algerian_Lamiaceae_v2/results/docking_results/redocking_validation/1EVE/1EVE_native_lig_redock.pdbqt  (6 rotatable branches)


## 6. Redocking (AutoDock Vina, screening settings)

In [7]:
def vina_dock(receptor, ligand, out, seed=VINA_SEED, exh=EXHAUSTIVENESS, n_poses=N_POSES):
    v = Vina(sf_name="vina", seed=seed, verbosity=0)
    v.set_receptor(receptor); v.set_ligand_from_file(ligand)
    v.compute_vina_maps(center=C, box_size=S)
    v.dock(exhaustiveness=exh, n_poses=n_poses)
    v.write_poses(out, n_poses=n_poses, overwrite=True)
    return [float(e[0]) for e in v.energies(n_poses=n_poses)]

redock_out = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_redock_out.pdbqt")
print(f"Redocking {NATIVE_SEL} into {PDB_CODE} (seed {VINA_SEED}, exhaustiveness {EXHAUSTIVENESS}) ...")
scores = vina_dock(RECEPTOR_PDBQT, native_pdbqt, redock_out)
print("Vina scores (kcal/mol):", [round(s, 3) for s in scores])

Redocking resn E20 into 1EVE (seed 42, exhaustiveness 10) ...
Vina scores (kcal/mol): [-10.966, -10.838, -10.396, -10.39, -9.908, -9.647, -9.622, -9.49, -9.368, -9.34]


## 7. RMSD vs crystal pose (in place, heavy atoms, symmetry-aware)

In [8]:
template = Chem.MolFromSmiles(NATIVE_SMILES)

def with_bond_orders(pdb_block):
    """Heavy-atom RDKit molecule from a PDB block, bond orders assigned from the SMILES template."""
    m = Chem.MolFromPDBBlock(pdb_block, removeHs=False, sanitize=False, proximityBonding=True)
    m = Chem.RemoveHs(m, sanitize=False)
    try:
        return AllChem.AssignBondOrdersFromTemplate(template, m), True
    except Exception:
        return m, False

def pdb_blocks(path):
    """One PDB block per structure in a mol2/pdb/pdbqt file (OpenBabel)."""
    conv = ob.OBConversion(); conv.SetInAndOutFormats(os.path.splitext(path)[1][1:], "pdb")
    mol = ob.OBMol(); blocks = []
    ok = conv.ReadFile(mol, path)
    while ok:
        mol.DeleteHydrogens(); blocks.append(conv.WriteString(mol))
        mol = ob.OBMol(); ok = conv.Read(mol)
    return blocks

def mcs_inplace_rmsd(ref, probe):
    """Fallback: element-based MCS, all symmetry-equivalent matches, in place (no superposition)."""
    mcs = rdFMCS.FindMCS([ref, probe], atomCompare=rdFMCS.AtomCompare.CompareElements,
                         bondCompare=rdFMCS.BondCompare.CompareAny, ringMatchesRingOnly=False, timeout=10)
    patt = Chem.MolFromSmarts(mcs.smartsString)
    rm = ref.GetSubstructMatches(patt, uniquify=False, maxMatches=1000)
    pm = probe.GetSubstructMatch(patt)
    if not rm or not pm: return None, 0
    rc, pc = ref.GetConformer().GetPositions(), probe.GetConformer().GetPositions()
    best = min(np.sqrt(np.mean(np.sum((rc[list(r)] - pc[list(pm)]) ** 2, axis=1))) for r in rm)
    return round(float(best), 3), len(pm)

def rmsd_table(pose_file, crystal_file, energies):
    ref, ref_ok = with_bond_orders(pdb_blocks(crystal_file)[0])
    rows = []
    for i, blk in enumerate(pdb_blocks(pose_file)):
        pose, ok = with_bond_orders(blk)
        r_inplace = r_super = None; method = "template"
        if ok and ref_ok:
            try:
                r_inplace = round(rdMolAlign.CalcRMS(Chem.Mol(pose), ref), 3)      # in place
                r_super   = round(rdMolAlign.GetBestRMS(Chem.Mol(pose), ref), 3)   # superimposed (old method)
            except Exception:
                ok = False
        if r_inplace is None:
            r_inplace, n = mcs_inplace_rmsd(ref, pose); method = f"MCS ({n} atoms)"
        rows.append({"Pose": i + 1, "Vina_score_kcal_mol": round(energies[i], 3),
                     "RMSD_in_place_A": r_inplace, "RMSD_superimposed_A": r_super, "Atom_matching": method})
    return pd.DataFrame(rows)

redock_df = rmsd_table(redock_out, NATIVE_LIG, scores)
display(redock_df)

top = redock_df.iloc[0]                         # best-scoring pose = the pose the protocol would report
best_any = redock_df["RMSD_in_place_A"].min()
RESULT = "PASS" if top["RMSD_in_place_A"] is not None and top["RMSD_in_place_A"] < RMSD_CUTOFF else "FAIL"
print(f"Top-scoring pose: score {top['Vina_score_kcal_mol']} kcal/mol, "
      f"RMSD in place {top['RMSD_in_place_A']} Å (superimposed: {top['RMSD_superimposed_A']} Å)")
print(f"Lowest in-place RMSD among the {len(redock_df)} poses: {best_any} Å")
print(f"Redocking validation (top pose, in place, < {RMSD_CUTOFF} Å): {RESULT}")
print(f"Previously reported in the paper: {PAPER_RMSD}")
redock_df.to_csv(os.path.join(REDOCK_DIR, f"{PDB_CODE}_redocking_validation.csv"), index=False)

,Pose,Vina_score_kcal_mol,RMSD_in_place_A,RMSD_superimposed_A,Atom_matching
0,1,-10.966,1.055,0.845,template
1,2,-10.838,8.969,2.510,template
2,3,-10.396,1.815,1.180,template
3,4,-10.390,8.653,2.739,template
4,5,-9.908,2.989,1.624,template
5,6,-9.647,9.523,2.206,template
6,7,-9.622,4.657,1.559,template
7,8,-9.490,5.697,1.183,template
8,9,-9.368,9.362,2.353,template
9,10,-9.340,5.714,1.352,template


Top-scoring pose: score -10.966 kcal/mol, RMSD in place 1.055 Å (superimposed: 0.845 Å)
Lowest in-place RMSD among the 10 poses: 1.055 Å
Redocking validation (top pose, in place, < 2.0 Å): PASS
Previously reported in the paper: 0.812 Å (Table 15, value from the earlier GetBestRMS cell)


## 8. Seed robustness

The same redocking with different random seeds; a validated protocol should reproduce the crystal pose for most seeds.

In [9]:
seed_rows = []
for sd in SEEDS_CHECK:
    out = os.path.join(REDOCK_DIR, f"{PDB_CODE}_native_lig_redock_seed{sd}.pdbqt")
    en = vina_dock(RECEPTOR_PDBQT, native_pdbqt, out, seed=sd)
    d = rmsd_table(out, NATIVE_LIG, en)
    seed_rows.append({"Seed": sd, "Top_score_kcal_mol": d.iloc[0]["Vina_score_kcal_mol"],
                      "Top_pose_RMSD_in_place_A": d.iloc[0]["RMSD_in_place_A"],
                      "Lowest_RMSD_any_pose_A": d["RMSD_in_place_A"].min()})
    print(f"  seed {sd:5d}: top score {seed_rows[-1]['Top_score_kcal_mol']:.3f}, "
          f"top-pose RMSD {seed_rows[-1]['Top_pose_RMSD_in_place_A']} Å")
seed_df = pd.DataFrame(seed_rows)
n_pass = int((seed_df["Top_pose_RMSD_in_place_A"] < RMSD_CUTOFF).sum())
print(f"\nTop pose within {RMSD_CUTOFF} Å for {n_pass} of {len(seed_df)} seeds")
seed_df.to_csv(os.path.join(REDOCK_DIR, f"{PDB_CODE}_redocking_seed_robustness.csv"), index=False)
display(seed_df)

  seed    42: top score -10.966, top-pose RMSD 1.055 Å


  seed     1: top score -10.946, top-pose RMSD 1.042 Å


  seed     7: top score -10.919, top-pose RMSD 1.021 Å


  seed   123: top score -10.951, top-pose RMSD 1.039 Å
  seed  2024: top score -10.949, top-pose RMSD 1.03 Å

Top pose within 2.0 Å for 5 of 5 seeds


,Seed,Top_score_kcal_mol,Top_pose_RMSD_in_place_A,Lowest_RMSD_any_pose_A
0,42,-10.966,1.055,1.055
1,1,-10.946,1.042,1.042
2,7,-10.919,1.021,1.021
3,123,-10.951,1.039,1.039
4,2024,-10.949,1.030,1.030


## 9. Result for Table 15

In [10]:
summary = {"Target (PDB)": f"{TARGET} ({PDB_CODE})", "Native ligand": NATIVE_SEL, "Engine": "AutoDock Vina 1.2",
           "Top-pose score (kcal/mol)": top["Vina_score_kcal_mol"],
           "RMSD in place (Å)": top["RMSD_in_place_A"], "RMSD superimposed (Å)": top["RMSD_superimposed_A"],
           "Seeds passing": f"{n_pass}/{len(seed_df)}", "Result": RESULT}
pd.DataFrame([summary]).to_csv(os.path.join(REDOCK_DIR, f"{PDB_CODE}_table15_row.csv"), index=False)
for k, v in summary.items(): print(f"{k:28s} {v}")
print("\nIf 'RMSD in place' differs from the value in Table 15, update Table 15 with the in-place value.")

Target (PDB)                 AChE (1EVE)
Native ligand                resn E20
Engine                       AutoDock Vina 1.2
Top-pose score (kcal/mol)    -10.966
RMSD in place (Å)            1.055
RMSD superimposed (Å)        0.845
Seeds passing                5/5
Result                       PASS

If 'RMSD in place' differs from the value in Table 15, update Table 15 with the in-place value.
